# Lab 04: Fine-Tune an Instruction Model with LoRA

This practical fine-tunes a small instruction-following model to answer computer-shop questions. Run the notebook in Google Colab with a GPU runtime for practical training speed. It detects the available GPU and selects Qwen2.5-0.5B-Instruct when at least 8 GiB VRAM is available, otherwise SmolLM2-360M-Instruct. If there is no CUDA GPU, it falls back to standard LoRA on CPU (slow) and skips 4-bit quantization.


## 1. Install libraries and inspect hardware
The GPU accelerates the large matrix multiplications used in transformer training and provides fast memory for model weights, activations, and optimizer state. The first code cell records the GPU model and memory; save its output in the Lab README after running.

In [ ]:
%pip -q install -U transformers datasets peft trl accelerate bitsandbytes

import torch
from IPython.display import display

has_cuda = torch.cuda.is_available()
if has_cuda:
    gpu_name = torch.cuda.get_device_name(0)
    gpu_memory_gib = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    gpu_capability = torch.cuda.get_device_capability(0)
    use_qlora = gpu_capability >= (6, 0)  # bitsandbytes NF4/FP4 requires compute capability 6.0+
    print(f'GPU available: {gpu_name} ({gpu_memory_gib:.1f} GiB VRAM; compute capability {gpu_capability[0]}.{gpu_capability[1]})')
    print('QLoRA: enabled with 4-bit NF4 quantization' if use_qlora else 'QLoRA: disabled; this GPU is below the bitsandbytes NF4 minimum, using standard LoRA.')
else:
    gpu_name, gpu_memory_gib = 'No CUDA GPU detected', 0
    gpu_capability, use_qlora = None, False
    print('GPU available: No CUDA GPU detected; CPU LoRA will be slow.')
    print('QLoRA: disabled because CUDA is unavailable.')

MODEL_ID = (
    'Qwen/Qwen2.5-0.5B-Instruct'
    if has_cuda and gpu_memory_gib >= 8
    else 'HuggingFaceTB/SmolLM2-360M-Instruct'
)
print('Selected model:', MODEL_ID)

## 2. Create a small domain dataset
The 36 examples below are original computer-shop support questions and answers. The final eight rows are held out for evaluation and are not used for training. The separate prompts at the end are also absent from the training set.

In [ ]:
from datasets import Dataset

qa_pairs = [
    ('What does a computer shop sell?', 'A computer shop sells computers, components, peripherals, and related services.'),
    ('Can I get help choosing a laptop?', 'Yes. Tell us your budget and what you plan to use it for, and we can suggest suitable laptops.'),
    ('What is the difference between RAM and storage?', 'RAM is short-term working memory; storage keeps files and programs when the computer is off.'),
    ('Do you sell desktop computers?', 'Yes. We sell desktops for everyday use, study, creative work, and gaming.'),
    ('Can you upgrade my laptop memory?', 'We can check whether your laptop memory is upgradeable and quote the compatible RAM.'),
    ('How do I choose a graphics card?', 'Choose a graphics card based on your software, monitor resolution, power supply, case space, and budget.'),
    ('Do you install components?', 'Yes. We can install compatible components and test the computer after installation.'),
    ('What should I bring for a repair?', 'Bring the device, its charger, and a description of the problem. Back up important files first if possible.'),
    ('Can you recover deleted files?', 'We can assess the storage device, but recovery is not guaranteed. Stop using the device to avoid overwriting data.'),
    ('What is an SSD?', 'An SSD is a fast storage device with no moving parts, commonly used for operating systems and files.'),
    ('Do you sell keyboards?', 'Yes. We offer wired and wireless keyboards in several layouts and styles.'),
    ('Can I return an item?', 'Returns depend on the store policy and the item condition. Keep the receipt and contact the shop to confirm eligibility.'),
    ('How can I check whether a charger is compatible?', 'Match the device model, connector, voltage, and required wattage before using a charger.'),
    ('Do you provide a warranty?', 'Warranty coverage depends on the product and manufacturer. Check the product documents or ask us for its warranty terms.'),
    ('My computer is overheating. What should I do?', 'Turn it off if it becomes very hot, keep vents clear, and arrange a diagnostic check if overheating continues.'),
    ('Can you remove a virus?', 'We can inspect the computer and help remove malicious software. Avoid entering passwords on a device you suspect is infected.'),
    ('What is a motherboard?', 'The motherboard is the main circuit board that connects the processor, memory, storage, and other components.'),
    ('Do you build custom PCs?', 'Yes. We can help select compatible parts and assemble a custom PC for your workload and budget.'),
    ('How much storage should I get?', 'It depends on your files and applications. A larger SSD is useful for games, media, and creative projects.'),
    ('Can I connect two monitors?', 'Many computers support two monitors, but it depends on the graphics hardware and available display ports.'),
    ('What is a power supply unit?', 'A power supply unit converts wall power into the voltages computer components need.'),
    ('My laptop battery drains quickly. Can you help?', 'We can check battery health, power settings, and background apps, then advise whether the battery needs replacement.'),
    ('Do you sell webcams?', 'Yes. We sell USB webcams for video calls, classes, and streaming.'),
    ('How do I protect my files?', 'Keep a separate backup on an external drive or trusted cloud service and update it regularly.'),
    ('Can you set up a new computer?', 'We can help with initial setup, updates, account sign-in, and transferring files from an old computer.'),
    ('What is thermal paste?', 'Thermal paste helps transfer heat from the processor to its cooler.'),
    ('Can I use a gaming PC for school?', 'Yes. A gaming PC can handle schoolwork, though it may cost more and use more power than needed for basic tasks.'),
    ('Do you sell USB-C hubs?', 'Yes. Check that a hub supports your device and the ports or display features you need.'),
    ('Why is my computer slow?', 'Common causes include too many startup apps, low storage, limited memory, malware, or aging hardware. A diagnostic can identify the cause.'),
    ('Can you clean dust from a desktop?', 'Yes. We can clean internal dust safely and check that the fans are working properly.'),
    ('What is a CPU?', 'The CPU is the processor that executes instructions and coordinates many computer tasks.'),
    ('Should I shut down my computer every night?', 'Either shutdown or sleep is fine for most users. Restarting occasionally also helps apply updates.'),
    ('Can I add Wi-Fi to a desktop?', 'Usually. A compatible Wi-Fi card or USB adapter can add wireless networking.'),
    ('What information do you need to recommend a PC?', 'Share your budget, main applications, preferred screen size, and any portability or gaming needs.'),
    ('Do you repair printers?', 'We can advise on common setup issues and let you know whether printer repairs are available at the shop.'),
    ('How do I safely transport a desktop PC?', 'Shut it down, unplug cables, secure loose accessories, and protect the case from impact and moisture.'),
]
assert len(qa_pairs) >= 30
print(f'{len(qa_pairs)} Q&A pairs created')
train_pairs, eval_pairs = qa_pairs[:-8], qa_pairs[-8:]
print(f'Training examples: {len(train_pairs)} | Held-out evaluation examples: {len(eval_pairs)}')

from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def to_text(pair):
    question, answer = pair
    messages = [
        {'role': 'system', 'content': 'You are a helpful assistant for a computer shop. Give concise, accurate answers and do not invent store policies.'},
        {'role': 'user', 'content': question},
        {'role': 'assistant', 'content': answer},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=False)

train_ds = Dataset.from_dict({'text': [to_text(x) for x in train_pairs]})
eval_ds = Dataset.from_dict({'text': [to_text(x) for x in eval_pairs]})
print('Example formatted training record:\n', train_ds[0]['text'])

## 3. Compare the original model and configure LoRA / QLoRA
We first collect original-model answers to all held-out questions. Then we apply LoRA. On CUDA, the model is loaded in 4-bit NF4 for QLoRA. On CPU, it uses ordinary LoRA.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

quant_config = None
model_kwargs = {}
if use_qlora:
    compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=compute_dtype,
    )
    model_kwargs.update(quantization_config=quant_config, device_map='auto')
elif use_qlora:
    compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    model_kwargs.update(torch_dtype=compute_dtype, device_map='auto')
else:
    model_kwargs.update(torch_dtype=torch.float32)

base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **model_kwargs)
base_model.config.use_cache = True

def ask(model, question, max_new_tokens=100):
    messages = [
        {'role': 'system', 'content': 'You are a helpful assistant for a computer shop. Give concise, accurate answers and do not invent store policies.'},
        {'role': 'user', 'content': question},
    ]
    input_ids = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors='pt').to(model.device)
    was_training = model.training
    was_use_cache = model.config.use_cache
    model.eval()
    model.config.use_cache = True
    try:
        with torch.no_grad():
            output = model.generate(input_ids, max_new_tokens=max_new_tokens, do_sample=False, pad_token_id=tokenizer.eos_token_id)
        return tokenizer.decode(output[0][input_ids.shape[-1]:], skip_special_tokens=True).strip()
    finally:
        model.config.use_cache = was_use_cache
        model.train(was_training)

heldout_questions = [q for q, _ in eval_pairs]
original_answers = [ask(base_model, q) for q in heldout_questions]

new_questions = [
    'A customer needs a quiet computer for online classes under a limited budget. What details should we ask before recommending one?',
    'A desktop turns on but shows no image. What basic safe checks can the customer try?',
    'Can a shop guarantee that every deleted file can be recovered?',
]
original_new_answers = [ask(base_model, q) for q in new_questions]

from peft import LoraConfig, prepare_model_for_kbit_training
if use_qlora:
    base_model = prepare_model_for_kbit_training(base_model)
else:
    base_model.gradient_checkpointing_enable()
base_model.config.use_cache = False
lora_config = LoraConfig(
    r=8, lora_alpha=16, lora_dropout=0.05, bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
)
trainable = sum(p.numel() for p in base_model.parameters() if p.requires_grad)
total = sum(p.numel() for p in base_model.parameters())
print(f'Base parameters: {total:,}; currently trainable before adapter injection: {trainable:,}')

## 4. Supervised fine-tuning with TRL SFTTrainer
Training is intentionally short for this small teaching dataset. Increase epochs only after checking training time and overfitting.

In [ ]:
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir='computer-shop-lora-checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=2 if gpu_memory_gib >= 8 else 1,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=1,
    eval_strategy='epoch',
    save_strategy='no',
    report_to='none',
    max_length=512,
    gradient_checkpointing=True,
gradient_checkpointing_kwargs={'use_reentrant': False},
    fp16=bool(has_cuda and not torch.cuda.is_bf16_supported()),
    bf16=bool(has_cuda and torch.cuda.is_bf16_supported()),
    optim='paged_adamw_8bit' if use_qlora else 'adamw_torch',
)
trainer = SFTTrainer(
    model=base_model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    processing_class=tokenizer,
    peft_config=lora_config,
)
trainer.model.print_trainable_parameters()
train_result = trainer.train()
print(train_result)
print('Training metrics:', trainer.state.log_history)

## 5. Evaluate, compare held-out prompts, and save the adapter
The held-out questions were not part of training. We report evaluation loss/perplexity and compare original versus fine-tuned answers using the exact same prompts. A small custom dataset is for demonstrating the workflow; it does not guarantee factual or production-quality behavior.

In [ ]:
import math

metrics = trainer.evaluate()
eval_loss = metrics.get('eval_loss')
perplexity = math.exp(eval_loss) if eval_loss is not None and eval_loss < 20 else float('inf')
print('Evaluation metrics:', metrics)
print(f'Evaluation perplexity: {perplexity:.3f}')

finetuned_answers = [ask(trainer.model, q) for q in heldout_questions]
comparison = Dataset.from_dict({
    'held_out_question': heldout_questions,
    'reference_answer': [a for _, a in eval_pairs],
    'original_model_answer': original_answers,
    'fine_tuned_model_answer': finetuned_answers,
})
display(comparison.to_pandas())

# These additional prompts are also absent from training and held-out evaluation.
for question, original_answer in zip(new_questions, original_new_answers):
    print('\nQuestion:', question)
    print('Original:', original_answer)
    print('Fine-tuned:', ask(trainer.model, question))

adapter_dir = 'computer-shop-lora-adapter'
trainer.model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print('Saved LoRA adapter and tokenizer to:', adapter_dir)

# Save a compact run record. The actual GPU and final metrics are populated by this runtime.
import json
run_record = {
    'model_id': MODEL_ID, 'gpu': gpu_name, 'gpu_memory_gib': round(gpu_memory_gib, 2),
    'quantization': '4-bit NF4 QLoRA' if use_qlora else ('standard LoRA on GPU' if has_cuda else 'standard LoRA on CPU'),
    'training_examples': len(train_pairs), 'evaluation_examples': len(eval_pairs),
    'evaluation_metrics': metrics, 'perplexity': perplexity,
}
with open('fine_tuning_run_record.json', 'w', encoding='utf-8') as f:
    json.dump(run_record, f, indent=2)
print(run_record)

import shutil
shutil.copy('fine_tuning_run_record.json', adapter_dir)
archive_path = shutil.make_archive(adapter_dir, 'zip', root_dir=adapter_dir)
print('Adapter bundle:', archive_path)
try:
    from google.colab import files
    files.download(archive_path)
except ImportError:
    print('Download the adapter ZIP from the runtime files panel.')